In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
timesteps = 20         # number of times the input is presented to the network
v_th = 1.0
decay = 0.25           # membrane potential decay factor
learning_rate = 1e-3   # learning rate for the optimizer
batch_size = 64        # batch size for paralleltraining
epochs = 5             # number of training epochs

In [ ]:
@tf.custom_gradient             # define a custom activation function with a surrogate gradient
def spike_function(x):          # step function for forward pass, x is the V-Vth
    out = tf.cast(x > 0, tf.float32)     # tf.cast converts boolean to float (1.0 for True, 0.0 for False)

    def grad(dy):                             # surrogate gradient for backward pass
        return dy * (1 / (1 + tf.abs(x))**2)  # fast sigmoid surrogate (differentiable) approximation 1 / (1 + |x|)^2

    return out, grad

In [ ]:
def poisson_encoder(x): #firing probability is proportional to the pixel intensity
    return tf.cast(tf.random.uniform(tf.shape(x)) < x, tf.float32)

#tf.random.uniform generates random values in [0, 1), compare with pixel intensity x to create spikes
#tf.cast converts boolean to float (1.0 for True, 0.0 for False)

In [ ]:
class SNNLeNet(tf.keras.Model):  # create a subclass of tf.keras.Model as SNNLeNet to define the architecture of the spiking neural network
    def __init__(self):         # initialize the model architecture, self is the instance of the class which means we can access the layers and variables using self.layer_name
        super().__init__()      # call the parent constructor


# LeNet-5 architecture
# Convolution Layer 1

        self.conv1 = tf.keras.layers.Conv2D(6, 5, padding='valid')
# It uses 6 filters of size 5x5 with no padding ( so we loose some edge info)
# So the spatial size reduces.
# Input:  (batch_size, 28, 28, 1)
# Output: (batch_size, 24, 24, 6)

        self.pool1 = tf.keras.layers.AveragePooling2D(pool_size=2)
# This reduces spatial dimensions by taking average over 2x2 regions.
# Input:  (batch_size, 24, 24, 6)
# Output: (batch_size, 12, 12, 6)


# Convolution Layer 2
        self.conv2 = tf.keras.layers.Conv2D(16, 5, padding='valid')
# Uses 16 filters of size 5x5, again with no padding.
# Input:  (batch_size, 12, 12, 6)
# Output: (batch_size, 8, 8, 16)

        self.pool2 = tf.keras.layers.AveragePooling2D(pool_size=2)
# Further reduces spatial dimensions.
# Input:  (batch_size, 8, 8, 16)
# Output: (batch_size, 4, 4, 16)


# Flatten layer
        self.flatten = tf.keras.layers.Flatten()
# Converts 3D feature maps into a 1D vector.
# Input:  (batch_size, 4, 4, 16)
# Output: (batch_size, 256)


# Fully Connected Layers
        self.fc1 = tf.keras.layers.Dense(120)
# Input:  (batch_size, 256)
# Output: (batch_size, 120)

        self.fc2 = tf.keras.layers.Dense(84)
# Input:  (batch_size, 120)
# Output: (batch_size, 84)

        self.fc3 = tf.keras.layers.Dense(10)
# Final output layer with 10 neurons (for 10 classes in MNIST).
# Produces spike-based outputs for classification.
# Input:  (batch_size, 84)
# Output: (batch_size, 10)

    def call(self, x): # define the forward pass through the network, call is a special method that allows the model to be called like a function
        batch_size = tf.shape(x)[0]

        # Initialize membrane potentials for each layer to zero at the start of processing each input sample
        mem1 = tf.zeros((batch_size, 24, 24, 6))
        mem2 = tf.zeros((batch_size, 8, 8, 16))
        mem3 = tf.zeros((batch_size, 120))
        mem4 = tf.zeros((batch_size, 84))
        mem5 = tf.zeros((batch_size, 10))

        spike_out = tf.zeros((batch_size, 10))  # accumulate output spikes over time

# pooling layers do not have membrane potentials as they are non-spiking operations that simply reduce spatial dimensions

        for t in range(timesteps):       # loop over the number of timesteps to simulate the temporal dynamics of spiking neurons

            input_spike = poisson_encoder(x)

            # Conv1
            cur1 = self.conv1(input_spike)         #cur1 is the current generated by the convolution operation , cur1 formula is Summation of (input_spike * weight) + bias
            mem1 = decay * mem1 + cur1             # update membrane potential with decay and new current
            spike1 = spike_function(mem1 - v_th)   # generate spikes based on whether the membrane potential exceeds the threshold
            mem1 = mem1 * (1 - spike1)             # reset membrane potential to zero where spikes occur

            # Pool1
            spike1_pooled = self.pool1(spike1)    #its stored as a matrix of 0s and 1s where 1s indicate the presence of a spike in that region after pooling

            # Conv2
            cur2 = self.conv2(spike1_pooled)
            mem2 = decay * mem2 + cur2
            spike2 = spike_function(mem2 - v_th)
            mem2 = mem2 * (1 - spike2)

            # Pool2
            spike2_pooled = self.pool2(spike2)

            # Flatten
            flat = self.flatten(spike2_pooled)

            # FC1
            cur3 = self.fc1(flat)
            mem3 = decay * mem3 + cur3
            spike3 = spike_function(mem3 - v_th)
            mem3 = mem3 * (1 - spike3)

            # FC2
            cur4 = self.fc2(spike3)
            mem4 = decay * mem4 + cur4
            spike4 = spike_function(mem4 - v_th)
            mem4 = mem4 * (1 - spike4)

            # FC3 (output)
            cur5 = self.fc3(spike4)
            mem5 = decay * mem5 + cur5
            spike5 = spike_function(mem5 - v_th)
            mem5 = mem5 * (1 - spike5)

            spike_out += spike5 # accumulate output spikes over time

        return spike_out / timesteps   # return the average firing rate over the timesteps as the output of the network

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()  # load the MNIST dataset, which consists of 28x28 grayscale images

x_train = x_train / 255.0 # normalize pixel values to [0, 1] range for better training stability and to represent them as firing probabilities for the Poisson encoder
x_test = x_test / 255.0

x_train = np.expand_dims(x_train, -1) # to add time dimension for the Poisson encodeing
x_test = np.expand_dims(x_test, -1)

In [ ]:
model = SNNLeNet()                                      # create an instance of the SNNLeNet model
optimizer = tf.keras.optimizers.Adam(learning_rate)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) #from_logits=True indicates that the output of the model is not passed through a softmax activation

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)) # create a tf.data.Dataset from the training data for efficient batching and shuffling during training
train_dataset = train_dataset.shuffle(10000).batch(batch_size) # shuffle the dataset with a buffer size of 10000 and batch it according to the specified batch size

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}")

    for step, (images, labels) in enumerate(train_dataset):# enuma

        with tf.GradientTape() as tape:     #tf,gradient tape helps compute the gradients
            outputs = model(images)         # get the model's predictions for the current batch of images
            loss = loss_fn(labels, outputs) # compute the loss between the true labels and the model's predictions using the specified loss function

        grads = tape.gradient(loss, model.trainable_variables)           # tape computes the gradients of the loss with respect to the model's trainable variables (weights and biases)
        optimizer.apply_gradients(zip(grads, model.trainable_variables)) #optimiser changes the variables to minimize the loss, zip is used to pair each gradient with its corresponding variable for the optimizer to update

        if step % 100 == 0:
            print(f"Step {step}, Loss: {loss.numpy():.4f}")


Epoch 1
Step 0, Loss: 2.3026
Step 100, Loss: 1.6867
Step 200, Loss: 1.5702
Step 300, Loss: 1.5319
Step 400, Loss: 1.5341
Step 500, Loss: 1.4981
Step 600, Loss: 1.5042
Step 700, Loss: 1.5607
Step 800, Loss: 1.4737
Step 900, Loss: 1.4776

Epoch 2
Step 0, Loss: 1.4808
Step 100, Loss: 1.4768
Step 200, Loss: 1.5210
Step 300, Loss: 1.4884
Step 400, Loss: 1.4828
Step 500, Loss: 1.4864
Step 600, Loss: 1.4841
Step 700, Loss: 1.4748
Step 800, Loss: 1.4948
Step 900, Loss: 1.5047

Epoch 3
Step 0, Loss: 1.5144
Step 100, Loss: 1.4767
Step 200, Loss: 1.4685
Step 300, Loss: 1.4688
Step 400, Loss: 1.4833
Step 500, Loss: 1.5069
Step 600, Loss: 1.4773
Step 700, Loss: 1.4969
Step 800, Loss: 1.4618
Step 900, Loss: 1.4727

Epoch 4
Step 0, Loss: 1.4810
Step 100, Loss: 1.4633
Step 200, Loss: 1.4633
Step 300, Loss: 1.4732
Step 400, Loss: 1.4681
Step 500, Loss: 1.4682
Step 600, Loss: 1.4699
Step 700, Loss: 1.4619
Step 800, Loss: 1.4800
Step 900, Loss: 1.4816

Epoch 5
Step 0, Loss: 1.4942
Step 100, Loss: 1.4680

In [ ]:
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(batch_size)

correct = 0
total = 0

for images, labels in test_dataset:
    outputs = model(images)

    preds = tf.argmax(outputs, axis=1, output_type=tf.int64)
    labels = tf.cast(labels, tf.int64) # labels are cast to int64 to match the data type of preds for accurate comparison

    correct += tf.reduce_sum(tf.cast(preds == labels, tf.int32)).numpy() #reduce_sum counts the number of correct predictions
    total += labels.shape[0]

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9851
